In [ ]:
import numpy as np
import json
import pandas as pd
from sklearn.model_selection import KFold 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from scipy.stats import kendalltau
import seaborn as sns
from tqdm import tqdm
import os

In [ ]:
lookback = 1
forecast_horizon = 1

In [ ]:
#Where the entire dataset split into its timestamps is stored
timestamps_directory = 'split_files_cleanedVEG/'

#Where the list of all timestamps are stored
timestamps_file_path = os.path.join(timestamps_directory, 'alltimestamps_cleanedVEG.json')

#Where the individual samples are stored
saved_files = 'lookback_and_lookahead_files_cleanedVEG/'

#Where the list of timestamps in their splits are stored
split_file = 'timestamps_splits_cleanedVEG.npz'

In [ ]:
#utility to load and sort all timestamps from the saved json file
#these timestamps are used to construct filenames for retrieving specific training, validation, and test samples
def load_all_timestamps():
    with open(timestamps_file_path, 'r') as file:
        timestamps = json.load(file)
        sorted_timestamps = sorted(timestamps)  #ensure the order is consistent
        return sorted_timestamps

#utility to return the length of the training set and the combined length of train + val
#used for indexing into the correct timestamp range
def load_split_lengths():
    loaded_data = np.load(split_file)
    train_len = len(loaded_data['train'])
    val_len = len(loaded_data['val'])
    return train_len, train_len + val_len  #used to calculate offset for val/test splits

#utility to return the individual sizes of the train, val, and test splits
def get_all_lengths():
    loaded_data = np.load(split_file)
    train_len = len(loaded_data['train'])
    val_len = len(loaded_data['val'])
    test_len = len(loaded_data['test'])
    return train_len, val_len, test_len

#function to load a single training sample given an index and lookback
#constructs the filename using the timestamp located at lookback + index
#returns the input-output pair stored in the npz file
def load_singular_train_data(index, lookback):
    all_timestamps = load_all_timestamps()
    timestamp_name = all_timestamps[lookback + index]
    file_name = f'{index + lookback}_{timestamp_name}.npz'
    file_path = os.path.join(saved_files, file_name)
    
    data = np.load(file_path, allow_pickle=True)
    return data['X_batches'], data['y_batches']

#function to load a single validation sample using index and lookback
#uses the offset of the training set to index into the validation range
def load_singular_val_data(index, lookback):
    all_timestamps = load_all_timestamps()
    train_len, _ = load_split_lengths()
    timestamp_name = all_timestamps[lookback + train_len + index]
    file_name = f'{index + train_len + lookback}_{timestamp_name}.npz'
    file_path = os.path.join(saved_files, file_name)
    
    data = np.load(file_path, allow_pickle=True)
    return data['X_batches'], data['y_batches']

#function to load a single test sample using index and lookback
#uses the offset of the training and validation sets to locate the test timestamp
def load_singular_test_data(index, lookback):
    all_timestamps = load_all_timestamps()
    train_len, train_val_len = load_split_lengths()
    timestamp_name = all_timestamps[lookback + train_val_len + index]
    file_name = f'{index + train_val_len + lookback}_{timestamp_name}.npz'
    file_path = os.path.join(saved_files, file_name)
    
    try:
        data = np.load(file_path, allow_pickle=True)
    except:
        print("the last erroneous files don't exist")

    return data['X_batches'], data['y_batches']


In [ ]:
#function to replace None values in a list with the mean of surrounding values using linear interpolation
def fill_none_with_mean(values):
    #convert the list to a numpy array, replacing None with np.nan
    array = np.array([np.nan if v is None else v for v in values])

    #identify the indices where the values are nan
    nan_indices = np.isnan(array)

    #get the indices and values of non-nan entries
    non_nan_indices = np.where(~nan_indices)[0]
    non_nan_values = array[non_nan_indices]

    #if all values are nan, return a list of zeros
    if len(non_nan_values) == 0: 
        return np.zeros_like(array).tolist()

    #use linear interpolation to fill the nan values based on surrounding valid values
    array[nan_indices] = np.interp(np.where(nan_indices)[0], non_nan_indices, non_nan_values)

    #return the filled array as a list
    return array.tolist()


In [ ]:
"""
This class defines a custom pytorch dataset for loading time series data stored on disk as .npz files.

Unlike traditional in-memory datasets, this class is used to load each training, validation, or test sample 
Directly from the hard drive using its index and mode (train/val/test). this approach helps:
- streamline training times by avoiding full memory loads
- keep training, validation, and test splits physically separated
- make it possible to work with datasets that are too large to fit into memory

Each sample corresponds to a pair of (X, y) arrays representing a lookback sequence and the next timestep target.
Missing values are filled using mean interpolation before converting to pytorch tensors.
"""

from torch.utils.data import Dataset, DataLoader

class TimeSeriesDataset(Dataset):
    def __init__(self, indices, lookback, mode="train"):
        """
        indices: list of data indices to be loaded from disk
        lookback: number of previous time steps to include in each input sample
        mode: one of 'train', 'val', or 'test' to determine which loader to use
        """
        self.indices = indices
        self.lookback = lookback
        self.mode = mode
    
    def __len__(self):
        #return the number of available samples
        return len(self.indices)
    
    def __getitem__(self, idx):
        #get the sample index
        index = self.indices[idx]

        #load the correct data split based on mode
        if self.mode == "train":
            X, y = load_singular_train_data(index, self.lookback)
        elif self.mode == "val":
            X, y = load_singular_val_data(index, self.lookback)
        elif self.mode == "test":
            X, y = load_singular_test_data(index, self.lookback)

        #fill missing values in each sub-array using linear mean interpolation
        for i, array in enumerate(X):
            for j, sub_array in enumerate(array):
                X[i][j] = fill_none_with_mean(sub_array)
        
        #convert to numpy array for tensor conversion
        X = np.array(X)

        #convert X and y to pytorch tensors
        X_tensor = torch.tensor(X, dtype=torch.float32)
        y_tensor = torch.tensor(y, dtype=torch.float32)
        
        return X_tensor, y_tensor


In [ ]:
#Getting lengths of the training, validation and testing
train_length, val_length, test_length = get_all_lengths()

In [ ]:
# Split data indices for train, validation, and test
train_length, val_length, test_length = get_all_lengths()

#We really should check this with training data
train_indices = list(range(0, train_length))
val_indices = list(range(0, val_length))
test_indices = list(range(0, test_length))

# Initialize Datasets
train_dataset = TimeSeriesDataset(train_indices, lookback, mode="train")
val_dataset = TimeSeriesDataset(val_indices, lookback, mode="val")
test_dataset = TimeSeriesDataset(test_indices, lookback, mode="test")

# Initialize DataLoaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
# Load global statistics from JSON file
import json

with open('globaldatastatistics_withVEG.json', 'r') as f:
    global_stats = json.load(f)

# Function to normalize data using global statistics
def normalize_global(batch, variable_name):
    mean = global_stats[variable_name]['mean']
    std = global_stats[variable_name]['std']
    return (batch - mean) / (std)

# Function to normalize data using local statistics
def normalize_local(batch):
    mean = batch.mean()  # Compute mean across the time dimension (axis=1)
    std = batch.std()# Compute std deviation across time dimension
    return (batch - mean) / (std)    # Normalize batch and leave a small epsilon to avoid division by 0

def normalize_precipitation(batch):
    log_normalized = torch.log(batch + 1)
    zero_indicator = (batch == 0).float()
    return log_normalized, zero_indicator


In [ ]:
#Variable names
variable_names = ['10 metre U wind component', '10 metre V wind component', '2 metre dewpoint temperature', '2 metre temperature', 'UV visible albedo for direct radiation (climatological)', 'Total column rain water', 'Volumetric soil water layer 1', 'Leaf area index, high vegetation', 'Leaf area index, low vegetation', 'Forecast surface roughness', 'Total precipitation', 'Time-integrated surface latent heat net flux', 'Evaporation']

In [ ]:
#MLP definition
class MLP_5D(nn.Module):
    def __init__(self, height, width):
        super(MLP_5D, self).__init__()
        # Define the fully connected layers
        self.fc1 = nn.Linear(64, 128)  # Input channels = 41, output features = 128
        self.dropout1 = nn.Dropout(0.05)
        self.fc2 = nn.Linear(128, 64)  # Output features = 64
        self.dropout2 = nn.Dropout(0.05)
        self.fc3 = nn.Linear(64, 1)    # Final output, reducing to 1 channel

        self.height = height
        self.width = width

    def forward(self, x):
        batch_size, timesteps, channels, height, width = x.shape
        
        # Ensure the input spatial dimensions match the expected height and width
        assert height == self.height and width == self.width, "Height and width mismatch"
        
        # Reshape to (batch * timesteps * height * width, channels)
        x = x.permute(0, 1, 3, 4, 2).reshape(-1, channels)
        # print(x.shape)
        
        # Apply MLP (Fully connected layers)
        x = self.fc1(x)
        x = torch.nn.functional.softplus(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = torch.nn.functional.softplus(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        x = torch.nn.functional.softplus(x)
        
        # Reshape back to (batch, timesteps, 1, height, width)
        x = x.view(batch_size, timesteps, self.height, self.width, 1).permute(0, 1, 4, 2, 3)

        return x

In [ ]:
  
#ConvLSTM definition
from ConvLSTM import ConvLSTM
import torch
import torch.nn as nn
from collections import defaultdict

class ConvLSTMNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dims, kernel_size, num_layers, output_channels, batch_first=True, pool_size=(2,2)):
        super(ConvLSTMNetwork, self).__init__()
        
        # ConvLSTM module
        self.convlstm = ConvLSTM(input_dim=input_dim,
                                 hidden_dim=hidden_dims,
                                 kernel_size=kernel_size,
                                 num_layers=num_layers,
                                 batch_first=batch_first,
                                 bias=True,
                                 return_all_layers=True)
        
        # Batch Normalization for each ConvLSTM layer's output
        self.batch_norms = nn.ModuleList([
            nn.BatchNorm3d(hidden_dim) for hidden_dim in hidden_dims
        ])

        # Final Conv3D layer for regression pathway
        self.conv3d = nn.Conv3d(in_channels=hidden_dims[-1],
                                out_channels=output_channels,
                                kernel_size=(1, 3, 3),
                                padding=(0, 1, 1))

        # MLP for regression output: (B,T,C,H,W) -> (B,T,1,H,W)
        self.mlp = MLP_5D(height=81, width=97)

        self.classification_head = nn.Sequential(
            nn.Conv3d(output_channels, 1, kernel_size=(1,1,1)),  # from C to 1 channel
            nn.Sigmoid()
        )

        self.activation_variance = defaultdict(list)

    def forward(self, x):
        """
        x: (B, T, input_dim, H, W)
        """
        # Forward through ConvLSTM
        layer_output_list, last_state_list = self.convlstm(x)
        
        # Apply batch norms
        for i, output in enumerate(layer_output_list):
            # output: (B, T, C, H, W)
            output = output.permute(0, 2, 1, 3, 4)  # (B, C, T, H, W) for BatchNorm3d
            output = self.batch_norms[i](output)
            output = output.permute(0, 2, 1, 3, 4)  # back to (B, T, C, H, W)

            #Track variance across spatial dimensions for hooks with activation tracking 
            activation_variance = output.var(dim=(3, 4)).mean().item()
            self.activation_variance[f"ConvLSTM_layer_{i}"].append(activation_variance)

            layer_output_list[i] = output
        
        # Take output from the last ConvLSTM layer
        final_output = layer_output_list[-1]  # (B, T, C, H, W)

        # Pass through Conv3D: needs (B,C,T,H,W)
        final_output = final_output.permute(0, 2, 1, 3, 4)  # (B,C,T,H,W)
        final_output = self.conv3d(final_output)
        # Now final_output: (B, output_channels, T, H, W)

        # Return to (B,T,C,H,W) for MLP (regression)
        final_output_t = final_output.permute(0, 2, 1, 3, 4)  # (B,T,C,H,W)

        # Regression output
        regression_output = self.mlp(final_output_t)  # (B,T,1,H,W)

        # Classification output:
        # The classification head is defined for (B,C,T,H,W), so reorder again
        final_output_c = final_output  # still (B,output_channels,T,H,W)
        classification_output = self.classification_head(final_output_c)
        # classification_output: (B,1,T,H,W)

        # Permute classification output to match (B,T,1,H,W) format
        classification_output = classification_output.permute(0, 2, 1, 3, 4)  # (B,T,1,H,W)

        return regression_output, classification_output

In [ ]:
# Setting device
if torch.cuda.is_available():
    print("running on cuda")
    device = torch.device('cuda')
else:
    print("running on the cpu")
    device = torch.device('cpu')

In [ ]:
input_dim = 1
set_lookback = 1
set_forecast_horizon = 1

# Adjust input_dim and output_channels according to your data specifics
model = ConvLSTMNetwork(
    input_dim=14 * set_lookback, 
    hidden_dims=[14, 32, 64], 
    kernel_size=(3,3), 
    num_layers=3, 
    output_channels=64 * set_forecast_horizon, 
    batch_first=True
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=0.005)

# Define separate loss functions
loss_fn = nn.MSELoss()       # For regression output
bce_loss_fn = nn.BCELoss()       # For classification output

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer)

train_losses = []
val_losses = []

y_true = []
y_pred = []

train_losses_per_fold = []
val_losses_per_fold = []

height = 81
width = 97

num_epochs = 200
scaling_factor = 1


In [ ]:

def spatial_correlation(y_true, y_pred):
    # Flatten the tensors to work with them
    y_true_flat = y_true.view(-1).cpu()
    y_pred_flat = y_pred.view(-1).cpu()

    # Compute the numerator: sum(P * T)
    numerator = torch.sum(y_pred_flat * y_true_flat)

    # Compute the denominator: sqrt(sum(P^2) * sum(T^2))
    denominator = torch.sqrt(torch.sum(y_pred_flat ** 2) * torch.sum(y_true_flat ** 2))

    # Compute the correlation (add epsilon to avoid division by zero)
    correlation = numerator / (denominator)

    return correlation.item()

In [ ]:
#Definition of evaluation metrics
from scipy.stats import pearsonr, spearmanr
def nash_sutcliffe_efficiency(observed, predicted):
    # Ensure inputs are tensors on the CPU
    observed = observed.cpu()
    predicted = predicted.cpu()

    # Compute the numerator and denominator
    numerator = torch.sum((observed - predicted) ** 2)
    denominator = torch.sum((observed - torch.mean(observed)) ** 2)

    # Calculate NSE
    nse = 1 - (numerator / denominator)
    return nse.item()

from scipy.stats import pearsonr

def pearson_correlation(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return pearsonr(y_true, y_pred)[0]  # Return the correlation coefficient

from scipy.stats import spearmanr

def spearman_correlation(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return spearmanr(y_true, y_pred).correlation  # Return the Spearman correlation

def mse(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return torch.mean((y_true - y_pred) ** 2).item()

def mae(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return torch.mean(torch.abs(y_true - y_pred)).item()

def percentage_error(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()
    
    return 100 * torch.mean((y_pred - y_true) / (y_true + 1e-6)).item()

def percentage_bias(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    return 100 * torch.sum(y_pred - y_true) / (torch.sum(y_true) + 1e-6)

import torch.nn.functional as F

def earth_movers_distance(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    # Compute EMD using the Wasserstein distance (L1 distance)
    emd = torch.mean(torch.abs(torch.sort(y_pred)[0] - torch.sort(y_true)[0])).item()
    return emd

def kendall_tau(y_true, y_pred):
    y_true = y_true.view(-1).cpu().numpy()  # Flatten and move to CPU
    y_pred = y_pred.view(-1).cpu().numpy()  # Flatten and move to CPU
    
    return kendalltau(y_true, y_pred).correlation  # Return the Kendall Tau

def r2_score(y_true, y_pred):
    # Ensure inputs are tensors on the CPU
    y_true = y_true.cpu()
    y_pred = y_pred.cpu()

    ss_total = torch.sum((y_true - torch.mean(y_true)) ** 2)
    ss_residual = torch.sum((y_true - y_pred) ** 2)
    
    return 1 - (ss_residual / (ss_total + 1e-6)).item()

def spatial_correlation(y_true, y_pred):
    # Flatten the tensors to work with them
    y_true_flat = y_true.view(-1).cpu()
    y_pred_flat = y_pred.view(-1).cpu()

    # Compute the numerator: sum(P * T)
    numerator = torch.sum(y_pred_flat * y_true_flat)

    # Compute the denominator: sqrt(sum(P^2) * sum(T^2))
    denominator = torch.sqrt(torch.sum(y_pred_flat ** 2) * torch.sum(y_true_flat ** 2))

    # Compute the correlation (add epsilon to avoid division by zero)
    correlation = numerator / (denominator)

    return correlation.item()

In [ ]:
#We assume that the normalized datasets are already saved in the same directory. These datasets should be those for the MultiTask ConvLSTM w/ Veg

normalized_train_data = torch.load("normalized_train_data_MULTI_TASK_WITH_VEG_ORIGINAL.pth")
normalized_val_data = torch.load("normalized_train_data_MULTI_TASK_WITH_VEG_ORIGINAL.pth")
normalized_test_data = torch.load("normalized_train_data_MULTI_TASK_WITH_VEG_ORIGINAL.pth")


print("Loaded normalized data successfully!")


In [ ]:
import torch
import numpy as np
import json
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
from tqdm import tqdm

#plot three side-by-side maps: original, perturbed, and difference
def plot_spatial_map(before, after, title_before, title_after, title_diff, cmap='coolwarm'):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    im1 = axes[0].imshow(before, cmap=cmap, aspect='equal')
    axes[0].set_title(title_before)
    plt.colorbar(im1, ax=axes[0])

    im2 = axes[1].imshow(after, cmap=cmap, aspect='equal')
    axes[1].set_title(title_after)
    plt.colorbar(im2, ax=axes[1])

    im3 = axes[2].imshow(after - before, cmap=cmap, aspect='equal')
    axes[2].set_title(title_diff)
    plt.colorbar(im3, ax=axes[2])

    plt.show()

#summarise the mean change in each vegetation variable after pgd perturbation
def summarize_vegetation_change(before, after):
    for i, var in enumerate(vegetation_variable_indices):
        mean_before = before[i].mean().item()
        mean_after = after[i].mean().item()
        change = mean_after - mean_before
        print(f"{var:50s}  Before: {mean_before:.16f}, After: {mean_after:.16f}, Change: {change:+.16f}")

#visualise vegetation change maps for each variable before and after pgd
def plot_vegetation_change_maps(before, after):
    for i, var in enumerate(vegetation_variable_indices):
        vmin = min(before[i].min(), after[i].min())
        vmax = max(before[i].max(), after[i].max())

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        im1 = axes[0].imshow(before[i], cmap='YlGn', aspect='equal', vmin=vmin, vmax=vmax)
        axes[0].set_title(f"{var} (Before)")
        plt.colorbar(im1, ax=axes[0])

        im2 = axes[1].imshow(after[i], cmap='YlGn', aspect='equal', vmin=vmin, vmax=vmax)
        axes[1].set_title(f"{var} (After)")
        plt.colorbar(im2, ax=axes[1])

        im3 = axes[2].imshow(after[i] - before[i], cmap='coolwarm', aspect='equal')
        axes[2].set_title(f"{var} Change")
        plt.colorbar(im3, ax=axes[2])

        plt.tight_layout()
        plt.show()


#use power iteration to estimate the dominant direction of the gradient tensor across all time and grid cells
#this is used to guide perturbations in the direction of highest model sensitivity
def power_iteration(grad_tensor, num_iters=3):
    shape = grad_tensor.shape  # (B, T, C, H, W)
    B, T, C, H, W = shape

    #flatten to (BT, C, HW) and transpose for matrix-vector products
    flat_grad = grad_tensor.view(B * T, C, H * W).transpose(1, 2)  # (BT, HW, C)

    #initialize a random vector to start power iteration
    v = torch.randn(B * T, C, 1, device=grad_tensor.device)  # (BT, C, 1)

    for _ in range(num_iters):
        Av = torch.bmm(flat_grad, v)  # (BT, HW, 1): apply forward pass
        AtAv = torch.bmm(flat_grad.transpose(1, 2), Av)  # (BT, C, 1): apply backward pass
        v = AtAv / (AtAv.norm(dim=1, keepdim=True) + 1e-8)  # normalize to prevent exploding vectors

    #reshape back to original grad_tensor shape
    direction = v.view(B, T, C, 1, 1).expand_as(grad_tensor)

    #normalize direction vector for use in pgd step
    direction = direction / (direction.norm() + 1e-8)

    return direction, v.squeeze(-1)  #return full tensor and squeezed dominant vector


#plot the mean absolute contribution of each vegetation variable to the dominant gradient direction
def plot_variable_contributions(dominant_vec, variable_names):
    mean_contrib = dominant_vec.abs().mean(dim=0).cpu().numpy()  #take average across all samples
    plt.figure(figsize=(16, 10))
    plt.bar(variable_names, mean_contrib, color='teal')
    plt.title("Relative Contributions of Vegetation Variables to Dominant PGD Direction")
    plt.ylabel("Mean Absolute Value in Dominant Vector")
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.grid(True)
    plt.show()

# -------------------------
# Model & Data Setup
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ConvLSTMNetwork(
    input_dim=14,  
    hidden_dims=[14, 32, 64],
    kernel_size=(3,3),
    num_layers=3,
    output_channels=64,
    batch_first=True
).to(device)

checkpoint = torch.load("NEWPIPELINEConvLSTM_MULTI_TASK_WITH_VEG_ORIGINAL", map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

print(" Model loaded successfully!")

# -------------------------
# Variable Names
# -------------------------
variable_names = [
    '10 metre U wind component', '10 metre V wind component', '2 metre dewpoint temperature',
    '2 metre temperature', 'UV visible albedo for direct radiation (climatological)',
    'Total column rain water', 'Volumetric soil water layer 1', 'Leaf area index, high vegetation',
    'Leaf area index, low vegetation', 'Forecast surface roughness', 'Total precipitation',
    'Time-integrated surface latent heat net flux', 'Evaporation'
]

vegetation_variable_indices = [
    'Leaf area index, high vegetation',
    'Leaf area index, low vegetation',
    'Forecast surface roughness',
    'Volumetric soil water layer 1',
    'UV visible albedo for direct radiation (climatological)',
    'Evaporation'
]

veg_indices = [variable_names.index(v) for v in vegetation_variable_indices]

lai_vars = [
    'Leaf area index, high vegetation',
    'Leaf area index, low vegetation'
]
lai_indices = [vegetation_variable_indices.index(v) for v in lai_vars]

#retrieving the array indices for our vegetation variables
evap_idx = vegetation_variable_indices.index("Evaporation")
albedo_idx = vegetation_variable_indices.index("UV visible albedo for direct radiation (climatological)")
soil_idx = vegetation_variable_indices.index("Volumetric soil water layer 1")

other_indices = [i for i in range(len(vegetation_variable_indices)) if i not in lai_indices]

#retriving our global data statistics
with open("globaldatastatistics_withVEG.json", "r") as f:
    global_stats = json.load(f)

#retrieving the minimum and maximum values for our vegetation variables
veg_min_values = torch.tensor(
    [global_stats[var]["min"] for var in vegetation_variable_indices], dtype=torch.float32
).to(device)

veg_max_values = torch.tensor(
    [global_stats[var]["max"] for var in vegetation_variable_indices], dtype=torch.float32
).to(device)

#loading per-grid bounds for each variable
with open("grid_min_max.json", "r") as f:
    grid_min_max = json.load(f)

grid_shape = (81, 97)
min_grids = [None] * len(vegetation_variable_indices)
max_grids = [None] * len(vegetation_variable_indices)

#for each lai variable, use per-grid min/max bounds from the JSON file (for more accurate masking over land/ocean)
for i in lai_indices:
    var = vegetation_variable_indices[i]
    min_grids[i] = torch.tensor(grid_min_max["min_values"][var], dtype=torch.float32).reshape(81, 97)
    max_grids[i] = torch.tensor(grid_min_max["max_values"][var], dtype=torch.float32).reshape(81, 97)

#for other vegetation variables, use global min/max bounds across all grid cells
for i in other_indices:
    min_grids[i] = torch.full(grid_shape, veg_min_values[i].item())
    max_grids[i] = torch.full(grid_shape, veg_max_values[i].item())

#stack all min/max bounds into tensors used to clamp values during PGD
grid_min_tensor = torch.stack(min_grids).to(device)
grid_max_tensor = torch.stack(max_grids).to(device)

#step size for projected gradient descent
alpha = 0.005

#maximum number of pgd iterations
num_iterations = 365

#minimum precipitation change (mm) to stop early
precip_threshold = 0.1

#minimum change in paf (%) to consider as significant (currently unused)
paf_threshold = 0.05

#standard mse loss used to drive pgd direction
loss_fn = torch.nn.MSELoss()

#custom directionality for vegetation variables: +1 means increase encouraged, -1 means decrease encouraged
veg_directions = torch.tensor([1, 1, 1, 1, -1, 1], dtype=torch.float32).to(device)

# -------------------------
# Load & Compute Gaussian Decay Mask
# -------------------------
#load hotspot indices (lat/lon) from json
def load_hotspot_data(hotspot_file):
    with open(hotspot_file, 'r') as f:
        hotspot_data = json.load(f)
    return np.array([(entry["grid_row"], entry["grid_col"]) for entry in hotspot_data])

#precompute a gaussian decay mask over the grid based on distance to hotspot locations
def precompute_gaussian_decay(grid_shape, hotspot_indices, sigma=2):
    lat_lon_grid = np.indices(grid_shape).reshape(2, -1).T
    distances = cdist(lat_lon_grid, hotspot_indices, metric='euclidean')
    min_distances = distances.min(axis=1).reshape(grid_shape)
    decay_mask = 0.5*(np.exp(-min_distances**2 / (2 * sigma**2)))  #gaussian decay kernel
    decay_mask = np.clip(decay_mask, 0.2, 1.0)  #avoid total suppression
    return torch.tensor(decay_mask, dtype=torch.float32).to(device)

hotspot_file = "hotspots.json"
hotspot_indices = load_hotspot_data(hotspot_file)
decay_mask = precompute_gaussian_decay(grid_shape, hotspot_indices)

#visualise the decay mask
plt.figure(figsize=(8, 6))
plt.imshow(decay_mask.cpu().numpy(), cmap='Reds', aspect='equal')
plt.title("Gaussian Decay Mask around Deforestation Hotspots")
plt.colorbar(label='Decay Weight')
plt.grid(False)
plt.xlabel("Longitude index")
plt.ylabel("Latitude index")
plt.show()

print(" Gaussian decay mask computed successfully!")

def compute_paf(precip, threshold=0.01):
    return (precip > threshold).float().mean().item()

monthly_dominant_vecs = []
monthly_avg_vectors = []
monthly_dominant_contributions = []

def projected_gradient_descent(
    model, X, y, veg_indices, veg_min_values, veg_max_values, alpha=0.005, num_iterations=365,
    precip_threshold=0.1, decay_mask=None
):
    X = X.to(device)
    y = y.to(device)

    #reshape to ensure proper spatial grid
    y = y.view(y.shape[0], y.shape[1], 1, 81, 97)
    X = X.view(X.shape[0], X.shape[1], 14, 81, 97)

    #initialize perturbed input (clone and track gradients)
    X_perturbed = X.clone().detach().to(device)
    X_perturbed.requires_grad = True

    dominant_vec_accumulator = []

    #record initial predictions and metrics
    with torch.no_grad():
        pred_output = model(X_perturbed)
        initial_precip = pred_output[0] if isinstance(pred_output, tuple) else pred_output
        initial_precip = initial_precip.detach()

    initial_paf = compute_paf(initial_precip)
    initial_mean = initial_precip.mean().item()

    #compute min and max change allowed for each vegetation variable, per hour
    veg_hourly_max_change = (((veg_max_values - veg_min_values)/2) / (365 * 24))
    veg_hourly_min_change = veg_hourly_max_change / 10
    veg_hourly_max_change = veg_hourly_max_change.to(device)

    iteration = 0
    significant_change = False

    while not significant_change and iteration < 1000:
        pred_output = model(X_perturbed)
        pred_precip = pred_output[0] if isinstance(pred_output, tuple) else pred_output

        loss = loss_fn(pred_precip, y)
        model.zero_grad()
        loss.backward()

        with torch.no_grad():
            gradients = X_perturbed.grad.detach().clone()
            veg_gradients = gradients[:, :, veg_indices, :, :]  #extract vegetation-specific gradients

            #compute dominant direction using power iteration
            direction, dominant_vec = power_iteration(veg_gradients)

            #---------analyse all gradient directions using svd (singular value decomposition)----------
            flat_grad = veg_gradients.view(-1, veg_gradients.shape[2], veg_gradients.shape[3] * veg_gradients.shape[4])
            flat_grad = flat_grad.transpose(1, 2)  #shape: (BT, HW, C)

            U, S, Vh = torch.linalg.svd(flat_grad, full_matrices=False)  #Vh gives direction vectors
            mean_S = S.mean(dim=0).cpu().numpy()  #singular values = gradient strength along principal directions

            #plot spectrum of singular values to interpret sensitivity
            plt.figure(figsize=(8, 5))
            plt.plot(range(1, len(mean_S)+1), mean_S, marker='o')
            plt.title("Singular Values of Gradient Matrix")
            plt.xlabel("Direction Index")
            plt.ylabel("Mean Singular Value (Importance)")
            plt.grid(True)
            plt.tight_layout()
            plt.show()

            #visualise top 3 contributing variables per principal direction
            k = 3
            for i in range(k):
                contrib = Vh[:, i, :].abs().mean(dim=0).cpu().numpy()
                plt.figure(figsize=(10, 4))
                plt.bar(vegetation_variable_indices, contrib)
                plt.title(f"Contribution of Vegetation Variables to Direction {i+1}")
                plt.ylabel("Mean Absolute Contribution")
                plt.xticks(rotation=25)
                plt.grid(True)
                plt.tight_layout()
                plt.show()

            #replace direction with 6th singular vector
            v6 = Vh[:, 5, :]
            mean_v6 = v6.mean(dim=0)
            expanded_v6 = mean_v6.view(1, 1, -1, 1, 1).expand_as(veg_gradients)
            direction = expanded_v6 / (expanded_v6.norm() + 1e-8)
            dominant_vec = mean_v6

            #--------------------apply pgd update step with constraints--------------------

            raw_step = alpha * direction
            expanded_max_delta = veg_hourly_max_change.view(1, 1, -1, 1, 1).expand_as(raw_step)
            expanded_min_delta = veg_hourly_min_change.view(1, 1, -1, 1, 1).expand_as(raw_step)

            clamped_step = torch.clamp(raw_step, -expanded_max_delta, expanded_max_delta)

            #enforce min delta threshold (for stability)
            too_small = clamped_step.abs() < expanded_min_delta
            clamped_step = torch.where(too_small, expanded_min_delta * clamped_step.sign(), clamped_step)

            #apply decay mask spatially (final smoothing)
            if decay_mask is not None:
                clamped_step = clamped_step * decay_mask.view(1, 1, 1, 81, 97)

            #apply update to vegetation values
            X_perturbed[:, :, veg_indices, :, :] += clamped_step

        #clear gradients for next iteration
        if X_perturbed.grad is not None:
            X_perturbed.grad.zero_()

        #compute metrics and check for stopping
        current_precip_mean = pred_precip.mean().item()
        delta_precip_mean = abs(current_precip_mean - initial_mean)
        significant_change = delta_precip_mean > precip_threshold

        dominant_vec_accumulator.append(dominant_vec)

        if significant_change or iteration == 999: 
            veg_vals = X_perturbed[:, :, veg_indices, :, :].detach()
            veg_deltas = (veg_vals - X[:, :, veg_indices, :, :]).detach()

            per_variable_change = veg_deltas.abs().mean(dim=(0, 1, 3, 4)).cpu().tolist()

            current_paf = compute_paf(pred_precip.detach())
            delta_paf = ((abs(current_paf - initial_paf))/initial_paf)*100
            overall_veg_change = veg_deltas.abs().mean().item()

            print(f" Δ Precipitation Mean: {delta_precip_mean:.16f} mm | Δ PAF: {delta_paf:.16f}% | Δ Vegetation (overall | abs mean): {overall_veg_change:.16f}")
            print(f" Current Precipitation Mean: {current_precip_mean:.16f} mm | Current PAF: {current_paf:.16f}%")
            print(f" Initial Precipitation Mean: {initial_mean:.16f} mm | Initial PAF: {initial_paf:.16f}%")

        iteration += 1

    before = X[0, 0, veg_indices].cpu().detach().numpy()
    after = X_perturbed[0, 0, veg_indices].cpu().detach().numpy()

    # -----------------PLOTTING FOR THE INDIVIDUAL RUNS-----------------------------

    summarize_vegetation_change(before, after)

    plot_vegetation_change_maps(before, after)

        # Visualise the contribution of each variable
    if len(dominant_vec_accumulator) > 0:
        dominant_vec_all = torch.stack(dominant_vec_accumulator, dim=0)
        avg_dominant_vec = dominant_vec_all.mean(dim=0)
        plot_variable_contributions(avg_dominant_vec, vegetation_variable_indices)

    print("this is the per varaible change")
    print(per_variable_change)

    return X_perturbed.detach(), dominant_vec_accumulator, per_variable_change

# -------------------------
# PGD on test data (aggregated analysis)
# -------------------------

#global storage for full analysis
all_veg_changes = []                  # stores avg monthly vegetation change across all months
all_precip_changes = []              # stores avg monthly precipitation change across all months
all_spatial_veg_maps = []           # stores avg monthly spatial maps of vegetation changes
all_spatial_precip_maps = []        # stores avg monthly spatial maps of precipitation changes

#monthly-level storage (reset every 720 hours)
monthly_veg_changes = []
monthly_precip_changes = []
monthly_spatial_veg_maps = []
monthly_spatial_precip_maps = []

#hourly-level storage (accumulates for fine-grained analysis)
hourly_veg_changes = []
hourly_precip_changes = []
hourly_spatial_veg_maps =[]
hourly_spatial_precip_maps = []
hourly_variable_veg_changes = []    # stores per-variable vegetation change after PGD per timestep

#dominant direction tracking
monthly_dominant_vecs = []          # singular vectors per PGD step
monthly_avg_vectors = []            # monthly-averaged dominant directions
monthly_dominant_contributions = [] # stores averaged contribution of each variable to monthly dominant direction
hourly_precip_deltas = []           # hourly precipitation delta (used for distribution plots)

# define how many hours are in a month (used for monthly aggregation)
hours_per_month = 30 * 24
hour_counter = 0

count = 0

for i, (X_val, y_val, _) in enumerate(tqdm(list(normalized_test_data), desc="Running PGD")):

    #optional early stopping for debugging
    if count >= 247:
        break 
    
    print(f"processing batch with shape {X_val.shape}")

    # run PGD to perturb vegetation inputs and get dominant gradient directions
    X_perturbed, dominant_vecs, per_variable_change = projected_gradient_descent(
        model, X_val, y_val, veg_indices, veg_min_values, veg_max_values,
        alpha, num_iterations, precip_threshold, decay_mask
    )

    # reshape inputs and outputs for consistent comparison
    X_val = X_val.view(X_val.shape[0], X_val.shape[1], 14, 81, 97).to(device)
    X_perturbed = X_perturbed.view(X_perturbed.shape[0], X_perturbed.shape[1], 14, 81, 97)
    y_val = y_val.view(y_val.shape[0], y_val.shape[1], 1, 81, 97).to(device)

    with torch.no_grad():
        #predict precipitation after PGD perturbation
        y_pred = model(X_perturbed)
        y_pred = y_pred[0] if isinstance(y_pred, tuple) else y_pred

        #predict original precipitation before PGD perturbation
        y_pred_before = model(X_val)
        y_pred_before = y_pred_before[0] if isinstance(y_pred_before, tuple) else y_pred_before

        #calculate change in predicted precipitation mean
        mean_before = y_pred_before.mean().item()
        mean_after = y_pred.mean().item()
        delta = mean_after - mean_before

    #compute mean vegetation difference across time and batch (result is per-grid)
    veg_diff = (X_perturbed[:, :, veg_indices, :, :] - X_val[:, :, veg_indices, :, :]).mean(dim=(0, 1))
    print(veg_diff.shape, "This is the shape of veg diff")
    monthly_spatial_veg_maps.append(veg_diff.cpu().numpy())

    #compute average precipitation difference across time and batch (result is 2D map)
    precip_diff = (y_pred - y_pred_before).mean(dim=(0, 1)).squeeze().cpu().detach().numpy()
    monthly_spatial_precip_maps.append(precip_diff)

    #compute spatial mean of vegetation and precipitation deltas
    mean_veg = veg_diff.mean().item()
    mean_precip = precip_diff.mean().item()

    #record monthly means
    monthly_veg_changes.append(mean_veg)
    monthly_precip_changes.append(mean_precip)

    #store singular vectors and PGD statistics
    monthly_dominant_vecs.extend(dominant_vecs)

    #store hourly-level stats
    hourly_veg_changes.append(mean_veg)
    hourly_precip_changes.append(mean_precip)
    hourly_spatial_veg_maps.append(veg_diff.cpu().numpy())
    hourly_spatial_precip_maps.append(precip_diff)
    hourly_variable_veg_changes.append(per_variable_change)
    hourly_precip_deltas.append(precip_diff.mean())

    hour_counter += 16  # assume each batch is ~16 hours of data

    if hour_counter >= hours_per_month:
        print("Aggregating for the month")

        # append monthly averages to global summary
        print(len(all_veg_changes))
        all_veg_changes.append(np.mean(monthly_veg_changes))
        all_precip_changes.append(np.mean(monthly_precip_changes))
        all_spatial_veg_maps.append(np.mean(monthly_spatial_veg_maps, axis=0))
        all_spatial_precip_maps.append(np.mean(monthly_spatial_precip_maps, axis=0))

        # compute average dominant direction over the month
        vecs_tensor = torch.stack(monthly_dominant_vecs)
        monthly_avg = vecs_tensor.mean(dim=0)
        monthly_avg_vectors.append(monthly_avg)

        # store variable-level contributions to monthly direction
        contrib = monthly_avg.cpu().numpy()
        monthly_dominant_contributions.append(contrib)

        hourly_variable_veg_changes.append(per_variable_change)  #record final PGD vector changes

        # reset monthly trackers
        monthly_veg_changes = []
        monthly_precip_changes = []
        monthly_spatial_veg_maps = []
        monthly_spatial_precip_maps = []
        hour_counter = 0

    count += 1

#convert to array for analysis and plotting
monthly_dominant_contributions = np.array(monthly_dominant_contributions)


In [ ]:
#ensure all entries in dominant contribution array are 1D vectors (flatten if batch-dim present)
monthly_dominant_contributions_fixed = []
for entry in monthly_dominant_contributions:
    entry = np.asarray(entry)
    if entry.ndim == 2:  # if multiple directions per batch, take mean
        monthly_dominant_contributions_fixed.append(entry.mean(axis=0))
    elif entry.ndim == 1:
        monthly_dominant_contributions_fixed.append(entry)
    else:
        raise ValueError(f"Unexpected shape: {entry.shape}")
monthly_dominant_contributions = np.stack(monthly_dominant_contributions_fixed)

#convert list of hourly variable-level vegetation changes to array: shape (hours, veg_vars)
veg_change_array = np.array(hourly_variable_veg_changes)

#compute mean PGD-induced change per variable
mean_change_per_variable = veg_change_array.mean(axis=0)

#display average impact per variable
print(" Mean vegetation change per variable across PGD runs:")
for var, mean_val in zip(vegetation_variable_indices, mean_change_per_variable):
    print(f"{var:<55} | Mean change: {mean_val:.8f}")

#plot how precipitation changes over time due to PGD
plt.plot(hourly_precip_deltas)
plt.xlabel("Hour Index")
plt.ylabel("Hourly Δ Precipitation (mm)")
plt.title("Hourly Change in Precipitation Due to PGD")
plt.grid(True)
plt.tight_layout()
plt.show()

#plot monthly contributions of each vegetation variable to dominant PGD direction
plt.figure(figsize=(18, 8))
for i, var in enumerate(vegetation_variable_indices):
    plt.plot(monthly_dominant_contributions[:, i], label=var)
plt.title("Dominant Direction Contributions Across Months")
plt.xlabel("Month")
plt.ylabel("Mean Contribution to PGD Direction")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

#reshape flattened spatial maps if needed
def fix_and_stack_maps(map_list, name, expected_shape=(81, 97)):
    fixed_maps = []
    for i, m in enumerate(map_list):
        m = np.asarray(m)
        if m.ndim == 1 and m.size == np.prod(expected_shape):
            m = m.reshape(expected_shape)
        elif m.shape != expected_shape:
            raise ValueError(f"{name} map at index {i} has invalid shape: {m.shape}")
        fixed_maps.append(m)
    return np.stack(fixed_maps)

#build average precipitation impact map
all_spatial_precip_maps_fixed = fix_and_stack_maps(all_spatial_precip_maps, "Precipitation")
global_precip_map = np.mean(all_spatial_precip_maps_fixed, axis=0)

#build global vegetation change maps for each variable
all_spatial_veg_maps = np.asarray(all_spatial_veg_maps)
global_veg_maps = all_spatial_veg_maps.mean(axis=0)

#plot mean precipitation response to PGD
plt.figure(figsize=(8, 6))
plt.imshow(global_precip_map, cmap='Blues', aspect='equal')
plt.title("Global Precipitation Impact Map")
plt.colorbar(label="Mean Precipitation Change")
plt.grid(False)
plt.show()

#Per-variable vegetation maps
for i, var in enumerate(vegetation_variable_indices):
    plt.figure(figsize=(8, 6))
    plt.imshow(global_veg_maps[i], cmap='YlOrBr', aspect='equal')
    plt.title(f"Global Sensitivity Map: {var}")
    plt.colorbar(label="Mean Vegetation Change")
    plt.grid(False)
    plt.tight_layout()
    plt.show()

#--- Veg vs Precip Scatter
plt.figure(figsize=(8, 5))
plt.scatter(all_veg_changes, all_precip_changes, color='darkgreen')
plt.xlabel("mean vegetation change")
plt.ylabel("mean precipitation change")
plt.title("vegetation change vs precipitation change (monthly)")
plt.grid(True)
plt.show()

#-------------------------
# Extra Statistics
#-------------------------

#calculate average change per vegetation variable
per_variable_changes = []
for i in range(len(vegetation_variable_indices)):
    changes = []
    for m in all_spatial_veg_maps:
        if m.ndim == 1:
            m = m.reshape(81, 97)
        changes.append(m.mean())
    per_variable_changes.append(np.mean(changes))

# Bar plot of mean veg changes per variable
plt.figure(figsize=(10, 5))
plt.bar(vegetation_variable_indices, per_variable_changes, color='olive')
plt.title("Average change per vegetation variable")
plt.ylabel("Mean change")
plt.xticks(rotation=20)
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

#Relationship between mean vegetation change and PAF change
monthly_paf_changes = np.abs(np.diff(all_precip_changes, prepend=all_precip_changes[0]))

plt.figure(figsize=(8, 5))
plt.plot(monthly_paf_changes, label='Monthly PAF change', color='royalblue')
plt.plot(all_veg_changes, label='Monthly Veg change', color='darkorange')
plt.xlabel("Month")
plt.ylabel("Change")
plt.title("PAF vs Vegetation Change Across Months")
plt.legend()
plt.grid(True)
plt.show()

# Summary Stats
print("\n--- Summary Statistics ---")
print(f"Average monthly vegetation change: {np.mean(all_veg_changes):.4f}")
print(f"Average monthly precipitation change: {np.mean(all_precip_changes):.4f}")
print(f"Average monthly PAF change: {np.mean(monthly_paf_changes):.4f}")
print("Average change per vegetation variable:")
print("Hourly variable veg changes", hourly_variable_veg_changes)
for var, val in zip(vegetation_variable_indices, per_variable_changes):
    print(f"  {var}: {val:.6f}")

plt.figure(figsize=(10, 5))
plt.plot(hourly_veg_changes, label="Hourly Vegetation Change", color='green')
plt.plot(hourly_precip_changes, label="Hourly Precipitation Change", color='blue')
plt.xlabel("Hour Index")
plt.ylabel("Change")
plt.title("Hourly Vegetation and Precipitation Changes")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

#convert lists to arrays if not already
hourly_precip_deltas = np.array(hourly_precip_deltas)
monthly_precip_changes = np.array(monthly_precip_changes)

# ---- BOX PLOTS ----
plt.figure(figsize=(12, 6))
plt.boxplot([hourly_precip_deltas, monthly_precip_changes], labels=["Hourly", "Monthly"])
plt.title("Box Plot of Precipitation Changes (Mean, not Absolute)")
plt.ylabel("Mean Precipitation Change (mm)")
plt.grid(True)
plt.tight_layout()
plt.show()

# ---- HISTOGRAMS ----
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(hourly_precip_deltas, bins=30, color='skyblue', edgecolor='black')
plt.title("Histogram of Hourly Precipitation Changes")
plt.xlabel("Precipitation Change (mm)")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
plt.hist(monthly_precip_changes, bins=15, color='lightgreen', edgecolor='black')
plt.title("Histogram of Monthly Precipitation Changes")
plt.xlabel("Precipitation Change (mm)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

#plot hourly correlation
plt.figure(figsize=(8, 5))
plt.scatter(hourly_veg_changes, hourly_precip_changes, color='darkorange')
plt.xlabel("Vegetation Change")
plt.ylabel("Precipitation Change")
plt.title("Hourly Vegetation vs Precipitation Change")
plt.grid(True)
plt.tight_layout()
plt.show()

#aggregate total monthly changes
print("\nFinal Monthly Summary:")
for i in range(len(all_veg_changes)):
    print(f"Month {i+1}: Total Veg Δ = {np.sum(hourly_veg_changes[i*hours_per_month//16:(i+1)*hours_per_month//16]):.6f}, "
          f"Total Precip Δ = {np.sum(hourly_precip_changes[i*hours_per_month//16:(i+1)*hours_per_month//16]):.6f}")

#plot vegetation vs precipitation change per month
plt.figure(figsize=(10, 5))
plt.plot([np.sum(hourly_veg_changes[i*hours_per_month//16:(i+1)*hours_per_month//16]) for i in range(len(all_veg_changes))], label='Total Monthly Vegetation Change')
plt.plot([np.sum(hourly_precip_changes[i*hours_per_month//16:(i+1)*hours_per_month//16]) for i in range(len(all_precip_changes))], label='Total Monthly Precip Change')
plt.xlabel("Month")
plt.ylabel("Total Change")
plt.title("Vegetation Change vs Hourly Precipitation Impact")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

#hourly PGD summary statistics 
print("\n========== PGD Summary Statistics ==========")
print(f"Average Monthly Vegetation Change (mean): {np.mean(all_veg_changes):.6f}")
print(f"Average Monthly Precipitation Change (mean): {np.mean(all_precip_changes):.6f}")
print(f"Average Hourly Vegetation Change (mean): {np.mean(hourly_veg_changes):.6f}")
print(f"Average Hourly Precipitation Change (mean): {np.mean(hourly_precip_changes):.6f}")
print(f"Standard Deviation of Hourly Vegetation Change: {np.std(hourly_veg_changes):.6f}")
print(f"Standard Deviation of Hourly Precipitation Change: {np.std(hourly_precip_changes):.6f}")
print(f"Number of Months Analysed: {len(all_veg_changes)}")
print(f"Total Number of PGD Iterations (hours): {len(hourly_veg_changes)}")
print("===========================================\n")


def compute_lp_norm_grid(data_list, p):
    """
    Computes the L^p norm across variables for each grid cell.
    Expects data_list to be a list of arrays with shape [variables, H, W].
    Returns an [H, W] array with the L^p norm at each grid cell.
    """
    stacked = np.stack(data_list, axis=0)  # [time, vars, H, W]
    norms = np.linalg.norm(stacked, ord=p, axis=1)  # Apply L^p norm across variables
    mean_norm = np.mean(norms, axis=0)  # Average over time
    return mean_norm

p_norm = 6  #l^6 norm

print(len(hourly_spatial_veg_maps))

# Compute lpp norm grid maps
veg_norm_grid = compute_lp_norm_grid(hourly_spatial_veg_maps[10:], p=p_norm)
precip_norm_grid = compute_lp_norm_grid(hourly_spatial_precip_maps, p=1)

def plot_single_grid(data, title, cmap='viridis'):
    plt.figure(figsize=(8, 6))
    im = plt.imshow(data, cmap=cmap, aspect='equal')
    plt.colorbar(im)
    plt.title(title)
    plt.xlabel("Longitude Index")
    plt.ylabel("Latitude Index")
    plt.grid(False)
    plt.tight_layout()
    plt.show()

plot_single_grid(veg_norm_grid, f"Vegetation Perturbation L^{p_norm} Norm")


